### What goes into Notebook 01:
* stratified train/validation split from the training set only
* first XGBoost multiclass run on 34-class
* evaluate with accuracy, weighted F1, macro F1, and per-class report
* save:
  * label encoder
  * model
  * feature column list
  * validation metrics
* only after that, run once on the merged holdout test set

### A good practical starting point for XGBoost is:
* `objective="multi:softprob"`
* `eval_metric="mlogloss"`
* `tree_method="hist"`
* `device="cuda"`
* `n_estimators` around 300 to 600
* `max_depth` around 6 to 10
* `learning_rate` around 0.05 to 0.1
* `subsample` around 0.8
* `colsample_bytree` around 0.8

Then we tune from there.

### Cell 1 -- Imports

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math
import random
import re

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from xgboost import XGBClassifier

### Cell 2 -- Config

In [2]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

TRAIN_MANIFEST_PATH = Path("artifacts/train_manifest.csv")
MODEL_DIR = Path("artifacts/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "label_8"
MAX_ROWS_PER_CLASS = 50_000
CHUNKSIZE = 250_000
VAL_SIZE = 0.20
XGB_DEVICE = "cpu"

FEATURE_COLUMNS = [
    "Header_Length",
    "Protocol Type",
    "Time_To_Live",
    "Rate",
    "fin_flag_number",
    "syn_flag_number",
    "rst_flag_number",
    "psh_flag_number",
    "ack_flag_number",
    "ece_flag_number",
    "cwr_flag_number",
    "ack_count",
    "syn_count",
    "fin_count",
    "rst_count",
    "HTTP",
    "HTTPS",
    "DNS",
    "Telnet",
    "SMTP",
    "SSH",
    "IRC",
    "TCP",
    "UDP",
    "DHCP",
    "ARP",
    "ICMP",
    "IGMP",
    "IPv",
    "LLC",
    "Tot sum",
    "Min",
    "Max",
    "AVG",
    "Std",
    "Tot size",
    "IAT",
    "Number",
    "Variance",
]

### Cell 3 --- Feature List and Helpers

In [3]:
train_manifest = pd.read_csv(TRAIN_MANIFEST_PATH)

class_names = sorted(train_manifest[TARGET_COL].dropna().unique())
label_to_id = {label: i for i, label in enumerate(class_names)}
id_to_label = {i: label for label, i in label_to_id.items()}

print("Number of classes:", len(class_names))
print(class_names)

Number of classes: 8
['BENIGN', 'BRUTEFORCE', 'DDOS', 'DOS', 'MIRAI', 'RECON', 'SPOOFING', 'WEB']


### Cell 4 -- Balanced Row Sampler from Training Files

In [4]:
def sample_training_rows(
    manifest_df,
    feature_columns,
    target_col="label_34",
    max_rows_per_class=50_000,
    chunksize=250_000,
    seed=42,
):
    rng = np.random.default_rng(seed)
    counters = defaultdict(int)
    parts = []

    shuffled = manifest_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    for row in shuffled.itertuples(index=False):
        label = getattr(row, target_col)

        if pd.isna(label):
            continue

        remaining = max_rows_per_class - counters[label]
        if remaining <= 0:
            continue

        path = Path(row.path)

        for chunk in pd.read_csv(path, usecols=feature_columns, chunksize=chunksize):
            for col in feature_columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float32")

            chunk = chunk.replace([np.inf, -np.inf], np.nan).dropna()

            if len(chunk) == 0:
                continue

            take_n = min(remaining, len(chunk))

            if take_n < len(chunk):
                idx = rng.choice(len(chunk), size=take_n, replace=False)
                chunk = chunk.iloc[idx].copy()
            else:
                chunk = chunk.copy()

            chunk[target_col] = label
            parts.append(chunk)

            counters[label] += len(chunk)
            remaining = max_rows_per_class - counters[label]

            if remaining <= 0:
                break

    sampled_df = pd.concat(parts, ignore_index=True)
    return sampled_df, dict(counters)

### Cell 5 -- Build Sampled Training Frame

In [5]:
sampled_df, class_row_counts = sample_training_rows(
    train_manifest,
    FEATURE_COLUMNS,
    target_col=TARGET_COL,
    max_rows_per_class=MAX_ROWS_PER_CLASS,
    chunksize=CHUNKSIZE,
    seed=SEED,
)

print("Sampled shape:", sampled_df.shape)
print(pd.Series(class_row_counts).sort_values(ascending=False))

Sampled shape: (337892, 40)
MIRAI         50000
DDOS          50000
BENIGN        50000
DOS           50000
RECON         50000
SPOOFING      50000
WEB           24828
BRUTEFORCE    13064
dtype: int64


### Cell 6 -- encode labels and split

In [6]:
sampled_df = sampled_df[sampled_df[TARGET_COL].isin(class_names)].copy()

X = sampled_df[FEATURE_COLUMNS].astype("float32")
y = sampled_df[TARGET_COL].map(label_to_id).astype("int32")

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y,
)

print(X_train.shape, X_val.shape)

(270313, 39) (67579, 39)


### Cell 7 -- Train XGBoost

In [7]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(class_names),
    eval_metric="mlogloss",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    device=XGB_DEVICE,
    random_state=SEED,
    n_jobs=8,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

[0]	validation_0-mlogloss:1.93449
[50]	validation_0-mlogloss:0.48700
[100]	validation_0-mlogloss:0.37083
[150]	validation_0-mlogloss:0.34706
[200]	validation_0-mlogloss:0.33556
[250]	validation_0-mlogloss:0.32907
[300]	validation_0-mlogloss:0.32491
[350]	validation_0-mlogloss:0.32169
[400]	validation_0-mlogloss:0.31930
[450]	validation_0-mlogloss:0.31755
[500]	validation_0-mlogloss:0.31614
[550]	validation_0-mlogloss:0.31488
[599]	validation_0-mlogloss:0.31409


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,'cpu'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


### Validation Metrics 

In [8]:
val_pred = model.predict(X_val)

val_acc = accuracy_score(y_val, val_pred)
val_f1_weighted = f1_score(y_val, val_pred, average="weighted")
val_f1_macro = f1_score(y_val, val_pred, average="macro")

print("Validation accuracy:", val_acc)
print("Validation weighted F1:", val_f1_weighted)
print("Validation macro F1:", val_f1_macro)

report_df = pd.DataFrame(
    classification_report(
        y_val,
        val_pred,
        target_names=[id_to_label[i] for i in range(len(class_names))],
        output_dict=True,
        zero_division=0,
    )
).T

display(report_df)

Validation accuracy: 0.8830257920359875
Validation weighted F1: 0.881443965948847
Validation macro F1: 0.8434938707171931


,precision,recall,f1-score,support
BENIGN,0.709220,0.886100,0.787855,10000.000000
BRUTEFORCE,0.807357,0.495599,0.614181,2613.000000
DDOS,0.999300,0.999400,0.999350,10000.000000
DOS,0.999400,0.999000,0.999200,10000.000000
MIRAI,0.999700,0.999800,0.999750,10000.000000
RECON,0.826662,0.863200,0.844536,10000.000000
SPOOFING,0.895191,0.778100,0.832549,10000.000000
WEB,0.718096,0.628876,0.670531,4966.000000
accuracy,0.883026,0.883026,0.883026,0.883026
macro avg,0.869366,0.831259,0.843494,67579.000000


### Cell 9 -- Save Model and Metadata

In [9]:
joblib.dump(model, MODEL_DIR / "xgb_8class_baseline.joblib")

metadata = {
    "feature_columns": FEATURE_COLUMNS,
    "target_col": TARGET_COL,
    "class_names": class_names,
    "label_to_id": label_to_id,
    "max_rows_per_class": MAX_ROWS_PER_CLASS,
    "chunksize": CHUNKSIZE,
    "seed": SEED,
    "validation_accuracy": float(val_acc),
    "validation_f1_weighted": float(val_f1_weighted),
    "validation_f1_macro": float(val_f1_macro),
}

with open(MODEL_DIR / "xgb_8class_baseline_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved model and metadata.")

Saved model and metadata.


In [10]:
def canonicalize_label(raw_label):
    if pd.isna(raw_label):
        return np.nan

    s = str(raw_label).strip()
    if s == "" or s.upper() == "NAN":
        return np.nan

    s = s.replace(".pcap.csv", "").replace(".csv", "").replace(".pcap", "")

    if re.fullmatch(r"BenignTraffic\d*", s, flags=re.IGNORECASE):
        return "BENIGN"
    if s.lower() == "benign_final":
        return "BENIGN"

    s = s.upper()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"__+", "_", s).strip("_")

    alias_map = {
        "BENIGNTRAFFIC": "BENIGN",
        "BENIGN_FINAL": "BENIGN",
        "BACKDOOR_MALWARE": "BACKDOOR_MALWARE",
        "BROWSERHIJACKING": "BROWSERHIJACKING",
        "COMMANDINJECTION": "COMMANDINJECTION",
        "SQLINJECTION": "SQLINJECTION",
        "UPLOADING_ATTACK": "UPLOADING_ATTACK",
        "XSS": "XSS",
        "MITM_ARPSPOOFING": "MITM_ARPSPOOFING",
        "DNS_SPOOFING": "DNS_SPOOFING",
        "DICTIONARYBRUTEFORCE": "DICTIONARYBRUTEFORCE",
        "RECON_HOSTDISCOVERY": "RECON_HOSTDISCOVERY",
        "RECON_OSSCAN": "RECON_OSSCAN",
        "RECON_PINGSWEEP": "RECON_PINGSWEEP",
        "RECON_PORTSCAN": "RECON_PORTSCAN",
        "VULNERABILITYSCAN": "VULNERABILITYSCAN",
        "MIRAI_GREETH_FLOOD": "MIRAI_GREETH_FLOOD",
        "MIRAI_GREIP_FLOOD": "MIRAI_GREIP_FLOOD",
        "MIRAI_UDPPLAIN": "MIRAI_UDPPLAIN",
    }

    return alias_map.get(s, s)


def map_label_to_group(label_34):
    if pd.isna(label_34):
        return np.nan
    if label_34 == "BENIGN":
        return "BENIGN"
    if label_34.startswith("DDOS_"):
        return "DDOS"
    if label_34.startswith("DOS_"):
        return "DOS"
    if label_34.startswith("RECON_") or label_34 == "VULNERABILITYSCAN":
        return "RECON"
    if label_34.startswith("MIRAI_"):
        return "MIRAI"
    if label_34 in {"DNS_SPOOFING", "MITM_ARPSPOOFING"}:
        return "SPOOFING"
    if label_34 == "DICTIONARYBRUTEFORCE":
        return "BRUTEFORCE"
    if label_34 in {
        "BACKDOOR_MALWARE",
        "BROWSERHIJACKING",
        "COMMANDINJECTION",
        "SQLINJECTION",
        "UPLOADING_ATTACK",
        "XSS",
    }:
        return "WEB"
    return "UNKNOWN"


def map_label_to_binary(label_34):
    if pd.isna(label_34):
        return np.nan
    return "BENIGN" if label_34 == "BENIGN" else "MALICIOUS"

### Cell 10 -- test chunck iterator

In [11]:
TEST_ROOT = Path("../data/CIC_IOT_Dataset_2023")
test_files = sorted(TEST_ROOT.glob("Merged*.csv"))
    
print("TEST_ROOT:", TEST_ROOT.resolve())
print("Number of test files found:", len(test_files))
print("First 3 test files:", test_files[:3])

if len(test_files) == 0:
    raise RuntimeError("No test files found. Check TEST_ROOT.")

def iter_test_chunks(test_files, feature_columns, chunksize=250_000):
    usecols = feature_columns + ["Label"]

    for path in test_files:
        for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
            for col in feature_columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float32")

            chunk = chunk.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_columns)

            chunk["label_raw"] = chunk["Label"]
            chunk["label_34"] = chunk["label_raw"].map(canonicalize_label)
            chunk["label_8"] = chunk["label_34"].map(map_label_to_group)
            chunk["label_bin"] = chunk["label_34"].map(map_label_to_binary)

            yield chunk.drop(columns=["Label"])

TEST_ROOT: /workspace/data/CIC_IOT_Dataset_2023
Number of test files found: 63
First 3 test files: [PosixPath('../data/CIC_IOT_Dataset_2023/Merged01.csv'), PosixPath('../data/CIC_IOT_Dataset_2023/Merged02.csv'), PosixPath('../data/CIC_IOT_Dataset_2023/Merged03.csv')]


### Cell 11 -- incremental confusion matrix evaluation

In [12]:
def metrics_from_confusion_matrix(cm):
    total = cm.sum()
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    support = cm.sum(axis=1)

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp, dtype=float),
        where=(precision + recall) != 0,
    )

    accuracy = float(tp.sum() / total) if total > 0 else np.nan
    macro_f1 = float(np.mean(f1)) if len(f1) > 0 else np.nan
    weighted_f1 = float(np.average(f1, weights=support)) if support.sum() > 0 else np.nan

    per_class = pd.DataFrame({
        "label": [id_to_label[i] for i in range(len(class_names))],
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })

    return accuracy, macro_f1, weighted_f1, per_class


cm = np.zeros((len(class_names), len(class_names)), dtype=np.int64)
dropped_missing = 0
seen_rows = 0

for chunk in iter_test_chunks(test_files, FEATURE_COLUMNS, chunksize=250_000):
    valid = chunk[TARGET_COL].notna() & chunk[TARGET_COL].isin(class_names)
    dropped_missing += (~valid).sum()

    chunk = chunk.loc[valid].copy()
    if len(chunk) == 0:
        continue

    X_chunk = chunk[FEATURE_COLUMNS].astype("float32")
    y_true = chunk[TARGET_COL].map(label_to_id).to_numpy(dtype=np.int32)
    y_pred = model.predict(X_chunk).astype(np.int32)

    np.add.at(cm, (y_true, y_pred), 1)
    seen_rows += len(chunk)

print("Evaluated test rows:", seen_rows)
print("Dropped unlabeled/invalid test rows:", dropped_missing)

test_acc, test_macro_f1, test_weighted_f1, test_per_class = metrics_from_confusion_matrix(cm)

print("Test accuracy:", test_acc)
print("Test macro F1:", test_macro_f1)
print("Test weighted F1:", test_weighted_f1)

display(test_per_class.sort_values("support", ascending=False))

Evaluated test rows: 45018243
Dropped unlabeled/invalid test rows: 0
Test accuracy: 0.603891959977203
Test macro F1: 0.45990414451781214
Test weighted F1: 0.6728621396135707


,label,precision,recall,f1,support
2,DDOS,0.873116,0.622407,0.726747,32535697
3,DOS,0.380097,0.412512,0.395642,7746340
4,MIRAI,0.993211,0.990483,0.991845,2521551
0,BENIGN,0.804391,0.783385,0.793749,1051313
5,RECON,0.026421,0.360221,0.049231,661108
6,SPOOFING,0.678706,0.334948,0.448538,465914
7,WEB,0.030020,0.730187,0.057670,23798
1,BRUTEFORCE,0.131494,0.601501,0.215810,12522


In [13]:
test_per_class.to_csv(MODEL_DIR / "xgb_8class_test_per_class.csv", index=False)

cm_df = pd.DataFrame(
    cm,
    index=[id_to_label[i] for i in range(len(class_names))],
    columns=[id_to_label[i] for i in range(len(class_names))]
)
cm_df.to_csv(MODEL_DIR / "xgb_8class_test_confusion_matrix.csv")

test_metrics = {
    "test_accuracy": float(test_acc),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "evaluated_test_rows": int(seen_rows),
    "dropped_unlabeled_or_invalid_rows": int(dropped_missing),
}
with open(MODEL_DIR / "xgb_8class_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved test metrics, per-class report, and confusion matrix.")

Saved test metrics, per-class report, and confusion matrix.
